In [ ]:
import os
import json
import glob
import gc
import time
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

DATA_ROOT = "/kaggle/input/datasets/michaildarm/zeroer-benchamarks/"
OUTPUT_ROOT = "/kaggle/working/embeddings"

# Which passes to run below. Whole-record embeddings are already cached from a
# prior run; per-attribute is the new pass for Arm B2. Layout mirrors
# datasets/embeddings/{record,attribute}/ on the target machine (data.py:
# RECORD_DIR, ATTR_DIR).
ENCODE_RECORD     = False   # whole-record: embeddings/record/<dataset>/
ENCODE_ATTRIBUTES = True    # per-attribute: embeddings/attribute/<dataset>/

RECORD_ROOT = os.path.join(OUTPUT_ROOT, "record")
ATTR_ROOT   = os.path.join(OUTPUT_ROOT, "attribute")

os.makedirs(RECORD_ROOT, exist_ok=True)
os.makedirs(ATTR_ROOT, exist_ok=True)

# One line per embed_texts() call that actually ran (cache hits aren't
# timed -- nothing was created). Shared across both passes so a single file
# has the full picture.
TIMING_LOG = os.path.join(OUTPUT_ROOT, "embedding_times.txt")

In [ ]:
def merge_split_model(part_dirs, merge_dir):
    """Symlink all files from several part-directories into one flat dir."""
    os.makedirs(merge_dir, exist_ok=True)
    for part in part_dirs:
        # files may be nested (e.g. .../transformers/default/1), so walk
        for root, _, files in os.walk(part):
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(merge_dir, f)
                if not os.path.exists(dst):       # first occurrence wins
                    os.symlink(src, dst)
    return merge_dir

In [ ]:
PARTS = sorted(glob.glob("/kaggle/input/datasets/michaildarm/harrier-oss-v1-27b-p*"))
harrier_path = merge_split_model(PARTS, "/kaggle/tmp/harrier-oss-v1-27b")

PARTS = sorted(glob.glob("/kaggle/input/datasets/michaildarm/kalm-embedding-gemma3-12b-2511-p*"))
kalm_path = merge_split_model(PARTS, "/kaggle/tmp/kalm-embedding-gemma3-12b-2511")

PARTS = sorted(glob.glob("/kaggle/input/datasets/michaildarm/f2llm-v2-14b-p*"))
f2llm_path = merge_split_model(PARTS, "/kaggle/tmp/f2llm-v2-14b")

In [ ]:
MODELS = {
    "microsoft/harrier-oss-v1-27b": harrier_path,
    "tencent/KaLM-Embedding-Gemma3-12B-2511": kalm_path,
    "nvidia/llama-embed-nemotron-8b": "/kaggle/input/models/michaildarm/llama-embed-nemotron-8b/transformers/default/1",
    #"Qwen/Qwen3-Embedding-8B": "/kaggle/input/models/michaildarm/qwen3-embedding-8b/transformers/default/1",
    #"Qwen/Qwen3-Embedding-4B": "/kaggle/input/models/michaildarm/qwen3-embedding-4b/transformers/default/1",
    #"Octen/Octen-Embedding-8B": "/kaggle/input/models/michaildarm/octen-embedding-8b/transformers/default/1",
    #"codefuse-ai/F2LLM-v2-14B": f2llm_path,
    #"codefuse-ai/F2LLM-v2-8B": "/kaggle/input/models/michaildarm/f2llm-v2-8b/transformers/default/1",
    #"microsoft/harrier-oss-v1-0.6b": "/kaggle/input/models/michaildarm/harrier-oss-v1-0-6b/transformers/default/1",
    #"codefuse-ai/F2LLM-v2-4B": "/kaggle/input/models/michaildarm/f2llm-v2-4b/transformers/default/1",
}

BATCH_SIZE = 32
MAX_LENGTH = 512

In [ ]:
# Per-dataset attribute allow-list, mirroring utils.py DATASETS. `id` never
# feeds the model (on dblp_scholar the left ids encode authors/venue/year while
# the right ids are opaque hashes), and fodors_zagats' `class` is the
# ground-truth cluster label -- all 112 matched pairs share it, and no other
# cross-table pair does.
ATTRIBUTES = {
    "fodors_zagats":         ["name", "addr", "city", "phone"],
    "dblp_acm":              ["title", "authors", "venue", "year"],
    "dblp_scholar":          ["title", "authors", "venue", "year"],
    "abt_buy":               ["name", "description", "price"],
    "amazon_googleproducts": ["title", "description", "manufacturer", "price"],
}

# Right-table renames applied before attribute selection (utils.py rename_2).
RENAME_RIGHT = {
    "amazon_googleproducts": {"name": "title"},
}

datasets = []

for root, dirs, files in os.walk(DATA_ROOT):

    if "left.csv" in files and "right.csv" in files:

        name = os.path.basename(root)

        if name not in ATTRIBUTES:
            raise KeyError(f"no attribute list for dataset {name!r}")

        datasets.append({
            "name": name,
            "path": root,
            "left": os.path.join(root, "left.csv"),
            "right": os.path.join(root, "right.csv"),
        })

print(f"Found {len(datasets)} datasets")


def dataframe_to_text(df, attributes):

    missing = [c for c in attributes if c not in df.columns]

    if missing:
        raise KeyError(f"missing columns {missing}; have {list(df.columns)}")

    texts = []

    for row in df[attributes].itertuples(index=False):

        parts = []
        for c, v in zip(attributes, row):

            if pd.isna(v):
                continue

            v = str(v).strip()

            if not v:
                continue

            parts.append(f"{c}: {v}")

        texts.append(" | ".join(parts))

    return texts

In [ ]:
POOLING_STRATEGY = {
    "microsoft/harrier-oss-v1-27b": "last",
    "tencent/KaLM-Embedding-Gemma3-12B-2511": "last",
    "nvidia/llama-embed-nemotron-8b": "last",
    #"Qwen/Qwen3-Embedding-8B": "last",
    #"Qwen/Qwen3-Embedding-4B": "last",
    #"Octen/Octen-Embedding-8B": "last",
    #"codefuse-ai/F2LLM-v2-14B": "last",
    #"codefuse-ai/F2LLM-v2-8B": "last",
    #"microsoft/harrier-oss-v1-0.6b": "last",
    #"codefuse-ai/F2LLM-v2-4B": "last",
}

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def last_token_pooling(last_hidden_state, attention_mask):
    # The seq_lens index below finds the last real token only under RIGHT
    # padding. The Gemma3-based tokenizers (harrier-27b, KaLM-Gemma3) default to
    # padding_side='left' -- a GemmaTokenizer class default, not visible in the
    # repo's tokenizer_config.json -- where the real tokens sit at the end and
    # that index lands inside the pad block instead.
    if attention_mask[:, -1].all():           # nothing padded on the right
        return last_hidden_state[:, -1]

    seq_lens = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)

    return last_hidden_state[batch_idx, seq_lens]

@torch.inference_mode()
def embed_texts(
    texts,
    model,
    tokenizer,
    pooling_type="mean",
    batch_size=BATCH_SIZE,
):
    vectors = []

    for start in tqdm(range(0, len(texts), batch_size), leave=False):

        batch = texts[start:start+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        outputs = model(**inputs)

        if pooling_type == "mean":
            emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        elif pooling_type == "last":
            emb = last_token_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        else:
            raise ValueError(f"Unknown pooling type: {pooling_type}")

        # normalise in fp32; bf16's 8-bit mantissa is too coarse for the norm
        emb = torch.nn.functional.normalize(emb.float(), p=2, dim=1)

        vectors.append(emb.cpu().numpy())

    return np.concatenate(vectors, axis=0)


def timed_embed_texts(texts, model, tokenizer, log_fields, **kwargs):
    """embed_texts(), timed end-to-end and appended to TIMING_LOG.

    ``log_fields`` identifies the call (pass, dataset, model, attribute, side,
    ...); n_texts and seconds are added automatically. One line per call, so
    the log reflects encode time only -- not model load/cache-hit time.
    """
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    vectors = embed_texts(texts, model, tokenizer, **kwargs)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    fields = dict(log_fields, n_texts=len(texts), seconds=f"{elapsed:.3f}")
    line = " | ".join(f"{k}={v}" for k, v in fields.items())
    ts = time.strftime("%Y-%m-%dT%H:%M:%S")
    with open(TIMING_LOG, "a") as f:
        f.write(f"{ts} | {line}\n")
    print(f"  [{elapsed:.1f}s] {line}")

    return vectors

In [ ]:
# One-off fix for the stale Gemma3 cache (pooled at a pad position) and the
# fodors_zagats class-label leak -- both only ever affected whole-record
# embeddings. Scoped to RECORD_ROOT and behind its own flag: this cell sits
# above the per-attribute pass, and an unscoped rmtree(OUTPUT_ROOT) here would
# delete the ~490k per-attribute encodes too.
import shutil

CLEAR_RECORD_CACHE = False

if CLEAR_RECORD_CACHE and os.path.exists(RECORD_ROOT):
    shutil.rmtree(RECORD_ROOT)
    print("Cleared", RECORD_ROOT)

os.makedirs(RECORD_ROOT, exist_ok=True)

In [ ]:
if ENCODE_RECORD:
    for ds in datasets:

        ds_name = ds["name"]
        print("\n" + "=" * 100)
        print("DATASET:", ds_name)

        out_dir = os.path.join(RECORD_ROOT, ds_name)
        os.makedirs(out_dir, exist_ok=True)

        attributes = ATTRIBUTES[ds_name]

        # load dataset once
        left_df = pd.read_csv(ds["left"])
        right_df = pd.read_csv(ds["right"])

        if ds_name in RENAME_RIGHT:
            right_df = right_df.rename(columns=RENAME_RIGHT[ds_name])

        left_texts = dataframe_to_text(left_df, attributes)
        right_texts = dataframe_to_text(right_df, attributes)

        print("Attributes:", attributes)
        print("Example left text:", left_texts[0][:200])

        for model_name, model_path in MODELS.items():

            short_name = model_name.replace("/", "_")

            print("\n" + "-" * 80)
            print("Loading model:", model_name)

            left_cache_path = os.path.join(out_dir, f"{short_name}_left.npy")
            right_cache_path = os.path.join(out_dir, f"{short_name}_right.npy")

            # -------------------------
            # CACHE CHECK (LEFT)
            # -------------------------
            if os.path.exists(left_cache_path) and os.path.exists(right_cache_path):
                print("Cache exists → skipping")
                continue

            tokenizer = AutoTokenizer.from_pretrained(
                model_path,
                trust_remote_code=True
            )

            model = AutoModel.from_pretrained(
                model_path,
                trust_remote_code=True,
                dtype=torch.bfloat16,
                device_map="auto",
            )

            model.eval()

            pooling_type = POOLING_STRATEGY.get(model_name, "mean")
            print("Using pooling:", pooling_type, "| padding_side:", tokenizer.padding_side)

            # -------------------------
            # LEFT EMBEDDINGS
            # -------------------------
            if os.path.exists(left_cache_path):
                print("Left cache found")
                left_emb = np.load(left_cache_path)
            else:
                left_emb = timed_embed_texts(
                    left_texts,
                    model,
                    tokenizer,
                    log_fields=dict(phase="record", dataset=ds_name,
                                     model=model_name, side="left"),
                    pooling_type=pooling_type
                )

                np.save(left_cache_path, left_emb)
                print("Saved LEFT cache:", left_cache_path)

            # -------------------------
            # RIGHT EMBEDDINGS
            # -------------------------
            if os.path.exists(right_cache_path):
                print("Right cache found")
                right_emb = np.load(right_cache_path)
            else:
                right_emb = timed_embed_texts(
                    right_texts,
                    model,
                    tokenizer,
                    log_fields=dict(phase="record", dataset=ds_name,
                                     model=model_name, side="right"),
                    pooling_type=pooling_type
                )

                np.save(right_cache_path, right_emb)
                print("Saved RIGHT cache:", right_cache_path)

            print("Shape:", left_emb.shape, right_emb.shape)

            # cleanup per model
            del model
            del tokenizer
            del left_emb
            del right_emb
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
else:
    print("ENCODE_RECORD is False -> skipping whole-record pass")

Per attribute encoding

In [ ]:
ATTR_MODELS = [
    "microsoft/harrier-oss-v1-27b",
    "tencent/KaLM-Embedding-Gemma3-12B-2511",
    "nvidia/llama-embed-nemotron-8b",
    #"Qwen/Qwen3-Embedding-8B",
    #"Qwen/Qwen3-Embedding-4B",
    #"Octen/Octen-Embedding-8B",
    #"codefuse-ai/F2LLM-v2-14B",
    #"codefuse-ai/F2LLM-v2-8B",
    #"microsoft/harrier-oss-v1-0.6b",
    #"codefuse-ai/F2LLM-v2-4B"
]

# Attribute values are far shorter than a serialised record, so the batch can be
# bigger than BATCH_SIZE. Drop this first if the 12B/14B models OOM.
ATTR_BATCH_SIZE = 64

missing = [m for m in ATTR_MODELS if m not in MODELS]
if missing:
    raise KeyError(f"not in MODELS: {missing}")
print(f"{len(ATTR_MODELS)} models x per-attribute passes")


In [ ]:
def attribute_column(df, attribute):
    """(unique_texts, inverse, present) for one attribute of one table.

    Serialisation matches ``dataframe_to_text`` with a single-element attribute
    list -- the ``"{attr}: {value}"`` prefix is kept, so the encoder still sees
    which field it is reading.

    Two things the whole-record path never had to handle:

    * **Missing values.** A record with no value for this attribute cannot be
      embedded -- an empty string yields an arbitrary vector whose cosine
      against anything is noise. Those rows are excluded from the encode and
      left as an exact zero row, with ``present`` recording which. Downstream
      the feature must be forced to 0 for any pair where either side is absent,
      matching ``zeroer_features``' documented Magellan convention (NaN in
      either value -> NaN feature -> 0 after fillna). This is not a rare edge
      case: right ``year`` on dblp_scholar is 54% missing, right
      ``manufacturer`` on amazon_googleproducts 93%, left ``price`` on abt_buy
      61%.
    * **Duplicates.** Single attributes repeat heavily where whole records do
      not (venue, year, city, manufacturer, price). Encoding unique values only
      and scattering back cuts the corpus from 315,487 strings to 162,933 --
      48.4% -- with identical output.
    """
    if attribute not in df.columns:
        raise KeyError(f"missing column {attribute!r}; have {list(df.columns)}")

    s = df[attribute].astype("string").str.strip()
    present = (df[attribute].notna() & s.notna() & (s != "")).to_numpy()

    texts = (attribute + ": " + s[present]).to_numpy()
    uniq, inverse = np.unique(texts, return_inverse=True)   # sorted uniques
    return uniq.tolist(), inverse, present


def scatter(vectors, inverse, present, n_rows):
    """Unique-value vectors -> one row per record; absent rows stay zero."""
    full = np.zeros((n_rows, vectors.shape[1]), dtype=vectors.dtype)
    full[present] = vectors[inverse]
    return full


In [ ]:
if ENCODE_ATTRIBUTES:
    for ds in datasets:

        ds_name = ds["name"]
        attributes = ATTRIBUTES[ds_name]

        out_dir = os.path.join(ATTR_ROOT, ds_name)
        os.makedirs(out_dir, exist_ok=True)

        print("\n" + "=" * 100)
        print("DATASET:", ds_name, "| attributes:", attributes)

        left_df = pd.read_csv(ds["left"])
        right_df = pd.read_csv(ds["right"])
        if ds_name in RENAME_RIGHT:
            right_df = right_df.rename(columns=RENAME_RIGHT[ds_name])
        sides = {"left": left_df, "right": right_df}

        # ---- presence masks: model-independent, written once -------------------
        presence_path = os.path.join(out_dir, "presence.npz")
        if not os.path.exists(presence_path):
            np.savez_compressed(presence_path, **{
                f"{a}_{side}": attribute_column(df, a)[2]
                for side, df in sides.items() for a in attributes})
            print("Saved presence masks:", presence_path)

        for model_name in ATTR_MODELS:

            short_name = model_name.replace("/", "_")
            model_path = MODELS[model_name]

            todo = [(a, side) for a in attributes for side in ("left", "right")
                    if not os.path.exists(
                        os.path.join(out_dir, f"{short_name}_{a}_{side}.npy"))]

            if not todo:
                print(f"  {short_name}: all {len(attributes)*2} files cached -> skip")
                continue

            print("\n" + "-" * 80)
            print("Loading model:", model_name, f"({len(todo)} passes)")

            tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
            model = AutoModel.from_pretrained(
                model_path, trust_remote_code=True,
                dtype=torch.bfloat16, device_map="auto")
            model.eval()

            pooling_type = POOLING_STRATEGY.get(model_name, "mean")
            print("Using pooling:", pooling_type, "| padding_side:", tokenizer.padding_side)

            for attribute, side in todo:

                df = sides[side]
                uniq, inverse, present = attribute_column(df, attribute)

                print(f"  {attribute:12s} {side:5s} "
                      f"{len(df):6d} rows | {len(uniq):6d} unique | "
                      f"{int((~present).sum()):5d} absent")

                vectors = timed_embed_texts(
                    uniq, model, tokenizer,
                    log_fields=dict(phase="attribute", dataset=ds_name,
                                     model=model_name, attribute=attribute, side=side),
                    pooling_type=pooling_type,
                    batch_size=ATTR_BATCH_SIZE)

                full = scatter(vectors, inverse, present, len(df))

                path = os.path.join(out_dir, f"{short_name}_{attribute}_{side}.npy")
                np.save(path, full)

                del vectors, full
                gc.collect()

            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
else:
    print("ENCODE_ATTRIBUTES is False -> skipping per-attribute pass")

In [ ]:
# --- verify -----------------------------------------------------------------
# Row alignment with the source CSVs is the one invariant the downstream
# feature builder cannot check for itself, and a zero row must mean "absent",
# never "encoded badly".
for ds in datasets:

    ds_name = ds["name"]
    out_dir = os.path.join(ATTR_ROOT, ds_name)
    if not os.path.isdir(out_dir):
        continue

    presence = np.load(os.path.join(out_dir, "presence.npz"))
    n_rows = {"left": len(pd.read_csv(ds["left"])),
              "right": len(pd.read_csv(ds["right"]))}

    for model_name in ATTR_MODELS:
        short_name = model_name.replace("/", "_")

        for attribute in ATTRIBUTES[ds_name]:
            for side in ("left", "right"):

                path = os.path.join(out_dir, f"{short_name}_{attribute}_{side}.npy")
                if not os.path.exists(path):
                    print(f"MISSING {path}")
                    continue

                vec = np.load(path, mmap_mode="r")
                mask = presence[f"{attribute}_{side}"]

                assert len(vec) == n_rows[side], (
                    f"{path}: {len(vec)} rows vs {n_rows[side]} in {side}.csv")

                zero = ~np.asarray(vec).any(axis=1)
                assert (zero == ~mask).all(), (
                    f"{path}: zero rows disagree with the presence mask "
                    f"({zero.sum()} zero vs {(~mask).sum()} absent)")

    print(f"{ds_name}: OK")

In [ ]:
!zip -r test.zip /kaggle/working/embeddings